In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd


In [15]:
HEADING_PATTERN = re.compile(r"^.{0,80}:$|^[A-Z][A-Z \-/&]{2,}$")
BULLET_PATTERN = re.compile(r"^(?:[-•\u2022\*]|\d+\.|[a-zA-Z]\))\s*(.+)")
SENTENCE_SPLIT_PATTERN = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")
CLAUSE_SPLIT_PATTERN = re.compile(r"[;:\u2013\u2014\-]\s+")


def normalize_line(text: str) -> str:
    """Clean a single line from rubric text."""
    text = text.replace("\xa0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def segment_rubric(text: str) -> List[Dict[str, Any]]:
    """Split rubric text into sections with titles and bullet points."""
    sections: List[Dict[str, Any]] = []
    current_title = "General criteria"
    current_items: List[str] = []

    for raw_line in text.splitlines():
        line = normalize_line(raw_line)
        if not line:
            continue

        if HEADING_PATTERN.match(line):
            if current_items:
                sections.append({"title": current_title, "items": current_items})
            current_title = line.rstrip(": ")
            current_items = []
            continue

        bullet_match = BULLET_PATTERN.match(line)
        if bullet_match:
            current_items.append(bullet_match.group(1).strip())
        elif current_items:
            current_items[-1] = f"{current_items[-1]} {line}".strip()
        else:
            current_items.append(line)

    if current_items:
        sections.append({"title": current_title, "items": current_items})

    return sections


def explode_item(item: str) -> List[str]:
    """Break a rubric bullet/paragraph into shorter fragments."""
    fragments: List[str] = []
    for clause in CLAUSE_SPLIT_PATTERN.split(item):
        clause = clause.strip(" -.•;:")
        if not clause:
            continue
        sentences = SENTENCE_SPLIT_PATTERN.split(clause)
        for sentence in sentences:
            cleaned = sentence.strip(" -.•;:")
            if cleaned:
                fragments.append(cleaned)
    return fragments or [item.strip()]


def slice_key_points(items: List[str], limit: int = 4) -> List[str]:
    key_points: List[str] = []
    seen: set[str] = set()
    for item in items:
        for fragment in explode_item(item):
            normalized = fragment.lower()
            if normalized in seen:
                continue
            seen.add(normalized)
            key_points.append(fragment)
            if len(key_points) >= limit:
                break
        if len(key_points) >= limit:
            break
    return key_points


def generate_key_points_from_rubric(text: str, min_points: int = 2) -> List[Dict[str, Any]]:
    sections = segment_rubric(text)
    if not sections:
        consolidated_items = [normalize_line(line) for line in text.splitlines() if normalize_line(line)]
        if consolidated_items:
            sections = [{"title": "General criteria", "items": consolidated_items}]

    dataset: List[Dict[str, Any]] = []

    for section in sections:
        key_points = slice_key_points(section["items"])
        if len(key_points) < min_points:
            continue
        dataset.append({
            "section_title": section["title"],
            "key_points": key_points,
            "raw_items": section["items"],
        })

    return dataset


def flatten_key_points(sectioned_points: List[Dict[str, Any]]) -> List[str]:
    """Flatten sectioned key points into a single list."""
    combined: List[str] = []
    for entry in sectioned_points:
        combined.extend(entry["key_points"])
    return combined


## 1. Provide the rubric text
Paste rubric content into the cell below or adapt it to load from a file. Keep the structure (headings followed by bullet points) to get the best results.


In [3]:
# TODO: Replace this string with the actual rubric section you want to process.
#rubric_text = """
PERFORMANCE: ECONOMIC ANALYSIS
- Demonstrates understanding of key economic indicators and their implications.
- Connects policy decisions to market outcomes with relevant examples.

COMMUNICATION:
- Uses precise vocabulary appropriate for financial discussions.
- Presents data-driven arguments with supporting evidence.
- Responds to counterarguments respectfully and effectively.
""".strip()

# Optionally, load from a text file instead of hard-coding the string.
#

rubric_text = Path("generated_data/my_rubric.txt").read_text(encoding="utf-8")


SyntaxError: unterminated triple-quoted string literal (detected at line 16) (228370946.py, line 11)

In [17]:
# Alternative: uncomment and adjust if the rubric file lives elsewhere
# rubric_text = Path("/path/to/your/rubric.txt").read_text(encoding="utf-8")
rubric_text = Path("my_rubric.txt").read_text(encoding="utf-8")


## 2. Extract key points from the rubric


In [18]:
sections = segment_rubric(rubric_text)
print(f"Detected {len(sections)} rubric section(s).")
for section in sections:
    print(f"- {section['title']} ({len(section['items'])} item(s))")


Detected 1 rubric section(s).
- General criteria (1 item(s))


In [19]:
sections_with_points = generate_key_points_from_rubric(rubric_text)
sections_df = pd.DataFrame([
    {
        "section_title": entry["section_title"],
        "key_points": entry["key_points"],
    }
    for entry in sections_with_points
])

sections_df


,section_title,key_points
0,General criteria,[Inflation is the sustained rise in the genera...


## 3. Review key points (section view)


In [20]:
if sections_with_points:
    sample_section = sections_with_points[0]
    display(json.dumps(sample_section, indent=2, ensure_ascii=False))
else:
    print("No key points generated. Check the rubric text and formatting.")


'{\n  "section_title": "General criteria",\n  "key_points": [\n    "Inflation is the sustained rise in the general level of prices of goods and services over time, leading to a fall in the purchasing power of money",\n    "When prices increase, each unit of currency buys fewer goods and services than before",\n    "Inflation is usually measured by the percentage change in a price index such as the Consumer Price Index (CPI)",\n    "Moderate inflation is common in growing economies, as it encourages spending and investment"\n  ],\n  "raw_items": [\n    "Inflation is the sustained rise in the general level of prices of goods and services over time, leading to a fall in the purchasing power of money. When prices increase, each unit of currency buys fewer goods and services than before. Inflation is usually measured by the percentage change in a price index such as the Consumer Price Index (CPI). Moderate inflation is common in growing economies, as it encourages spending and investment. H

## 4. Optional: combine all key points


In [21]:
all_key_points = flatten_key_points(sections_with_points)
pd.DataFrame({"key_point": all_key_points})


,key_point
0,Inflation is the sustained rise in the general...
1,"When prices increase, each unit of currency bu..."
2,Inflation is usually measured by the percentag...
3,Moderate inflation is common in growing econom...


## 5. Export key points to JSON


In [23]:
output_path = Path("rubric_key_points.json")

if sections_with_points:
    export_payload = [
        {
            "section_title": entry["section_title"],
            "key_points": entry["key_points"],
        }
        for entry in sections_with_points
    ]
    output_path.write_text(json.dumps(export_payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved {len(export_payload)} section(s) to {output_path}.")
else:
    print("Nothing to export — please generate key points first.")


Saved 1 section(s) to rubric_key_points.json.


### Next steps
- Replace the placeholder rubric text with the full rubric you plan to analyze.
- If the rubric uses tables or nested bullets, tweak `segment_rubric` or `explode_item` to capture those structures.
- Rerun the notebook from the top after updates so the exported JSON stays current.
